### 欢迎来到第六周第三天！

让我们来试验更多不同的 MCP Server

In [ ]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio
import os
from IPython.display import Markdown, display
from datetime import datetime
load_dotenv(override=True)

### 第一种 MCP Server 类型：本地运行，一切都在本地

这是一个非常有趣的：基于知识图谱的记忆系统。

它是一个持久化的记忆存储，包含实体、对实体的观察以及实体之间的关系。

https://github.com/modelcontextprotocol/servers/tree/main/src/memory


In [ ]:
params = {"command": "npx","args": ["-y", "mcp-memory-libsql"],"env": {"LIBSQL_URL": "file:./memory/ed.db"}}

async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as server:
    mcp_tools = await server.list_tools()

mcp_tools

In [ ]:
instructions = "你使用实体工具作为持久化记忆来存储和回忆关于对话的信息。"
request = "我叫 Ed。我是一名 LLM 工程师。我正在教授一门关于 AI Agent 的课程，其中包括令人惊叹的 MCP 协议。\
MCP 是一个将 Agent 与工具、资源和提示模板连接起来的协议，使得 AI Agent 能够轻松集成各种能力。"
model = "gpt-4.1-mini"

In [ ]:
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

In [ ]:
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, "我叫 Ed。你对我了解多少？")
    display(Markdown(result.final_output))

### 查看 Trace 追踪：

https://platform.openai.com/traces

### 第二种 MCP Server 类型 — 本地运行，调用 Web 服务

### Brave Search — 抱歉 — 这需要另一个 API 密钥！但同样是免费的。

https://brave.com/search/api/

注册你的账户，然后将密钥添加到 .env 文件中的 `BRAVE_API_KEY`

In [ ]:
env = {"BRAVE_API_KEY": os.getenv("BRAVE_API_KEY")}
params = {"command": "npx", "args": ["-y", "@modelcontextprotocol/server-brave-search"], "env": env}

async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as server:
    mcp_tools = await server.list_tools()

mcp_tools

In [ ]:
instructions = "你能够搜索网络信息并简要总结要点。"
request = f"请研究亚马逊股价的最新消息，并简要总结其前景。\
作为参考，当前日期是 {datetime.now().strftime('%Y-%m-%d')}"
model = "gpt-4o-mini"

In [ ]:
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

### 和往常一样，查看 Trace 追踪：

https://platform.openai.com/traces

## 现在是第三种类型：远程运行

实际上很难找到"远程 MCP Server"，也叫"托管 MCP Server"或"管理型 MCP Server"。

这不是使用或分享 MCP Server 的常见模式，而且也没有标准的方式来发现远程 MCP Server。

Anthropic 列出了一些远程 MCP Server，但这些是面向付费应用的商业用户：

https://docs.anthropic.com/en/docs/agents-and-tools/remote-mcp-servers

CloudFlare 提供了工具让你创建和部署自己的远程 MCP Server，但这似乎并不是一种常见的做法：

https://developers.cloudflare.com/agents/guides/remote-mcp-server/


# 回到第二种类型：Polygon.io MCP Server

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">请务必阅读！！-</h2>
            <span style="color:#ff7800;">这个金融市场数据服务同时提供免费计划和付费计划，我们可以根据你的需求选择使用其中任何一种。
            </span>
        </td>
    </tr>
</table>

## 新章节：介绍 polygon.io

Polygon.io 是一个非常受欢迎的金融数据提供商。它有免费计划和付费计划。而且它还有一个 MCP Server！

首先，在他们优秀的网站上了解 polygon.io，包括查看定价：

https://polygon.io

### Polygon.io 第一部分：Polygon.io 免费服务（付费当然是完全可选的！）

1. 请注册 polygon.io（右上角）
2. 登录后，请在左侧导航栏中选择"Keys"
3. 点击蓝色的"New Key"按钮
4. 复制密钥名称
5. 编辑你的 .env 文件并添加以下行：

`POLYGON_API_KEY=xxxx`

In [ ]:
load_dotenv(override=True)
polygon_api_key = os.getenv("POLYGON_API_KEY")
if not polygon_api_key:
    print("POLYGON_API_KEY 未设置")

In [ ]:
from polygon import RESTClient
client = RESTClient(polygon_api_key)
client.get_previous_close_agg("AAPL")[0]

### 封装成一个缓存收盘价的 Python 模块

我制作了一个 Python 模块 `market.py`，使用这个 API 来查询股价。

但免费 API 的速率限制相当严格 — 所以我用了一点小技巧：当你查询股价时，这个函数会获取整个股票市场的收盘数据，并将其缓存到我们的数据库中。


In [ ]:
from market import get_share_price
get_share_price("AAPL")

In [ ]:
# 没有速率限制的担忧！

for i in range(1000):
    get_share_price("AAPL")
get_share_price("AAPL")

### 然后我把它做成了一个 MCP Server

就像我们对 accounts.py 所做的那样；参见 `market_server.py`

In [ ]:
params = {"command": "uv", "args": ["run", "market_server.py"]}
async with MCPServerStdio(params=params, client_session_timeout_seconds=60) as server:
    mcp_tools = await server.list_tools()
mcp_tools

### 让我们来试试！

希望 gpt-4o-mini 足够聪明，知道苹果的股票代码是 AAPL

In [ ]:
instructions = "你回答关于股票市场的问题。"
request = "苹果的股价是多少？"
model = "gpt-4.1-mini"

async with MCPServerStdio(params=params, client_session_timeout_seconds=60) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

## Polygon.io 第二部分：付费计划 — 完全可选！

如果你有兴趣，可以订阅月度计划以获取更实时的市场数据和无限 API 调用。

如果你确实想这样做，那么使用 Polygon.io 发布的完整 MCP Server 也是合理的，以便利用他们的所有功能。


In [ ]:

params = {"command": "uvx",
          "args": ["--from", "git+https://github.com/polygon-io/mcp_polygon@v0.1.0", "mcp_polygon"],
          "env": {"POLYGON_API_KEY": polygon_api_key}
          }
async with MCPServerStdio(params=params, client_session_timeout_seconds=60) as server:
    mcp_tools = await server.list_tools()
mcp_tools


### 哇，工具真多！

让我们来试试 — 希望工具数量不会让 gpt-4o-mini 不知所措！

使用 $29 的月度计划，我们无法访问某些 API，所以我需要指定哪些 API 可以被调用。

如果你升级到了更大的计划，可以随意移除我的额外限制..

In [ ]:
instructions = "你回答关于股票市场的问题。"
request = "苹果的股价是多少？使用你的 get_snapshot_ticker 工具获取最新价格。"
model = "gpt-4.1-mini"

async with MCPServerStdio(params=params, client_session_timeout_seconds=60) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

## 设置你的 .env 文件

如果你决定使用付费计划，请在 .env 文件中添加以下内容来表示：

`POLYGON_PLAN=paid`

如果你决定使用实时 API，请添加：

`POLYGON_PLAN=realtime`

In [ ]:
load_dotenv(override=True)

polygon_plan = os.getenv("POLYGON_PLAN")
is_paid_polygon = polygon_plan == "paid"
is_realtime_polygon = polygon_plan == "realtime"

if is_paid_polygon:
    print("你已选择订阅付费 Polygon 计划，代码将查看延迟 15 分钟的价格")
elif is_realtime_polygon:
    print("厉害 — 你已选择订阅实时 Polygon 计划，代码将查看实时价格")
else:
    print("根据你的 .env 文件，你已选择订阅免费 Polygon 计划，代码将查看收盘价格")

## 今天就到这里！

我移除了本实验中使用"Financial Datasets" MCP Server 的部分，因为它更贵且 API 更少。

这样我们可以使用同一个提供商来提供免费和付费 API。

但如果你想查看代码，只需在 git 历史中查找之前的版本。

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">练习</h2>
            <span style="color:#ff7800;">探索 MCP Server 市场，并使用所有 3 种方式来集成你自己的 MCP Server。
            </span>
        </td>
    </tr>
</table>